In [181]:
import numpy as np
import pandas as pd
import os
import sys
import seaborn as sns
import pickle
import matplotlib.pyplot as plt
import scipy.stats as stats 
import missingno as msno
import sklearn as sk
import optuna
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.impute import KNNImputer
from sklearn.preprocessing import PolynomialFeatures, RobustScaler, StandardScaler, PowerTransformer
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from feature_engine.outliers import Winsorizer
sys.path.append("../src/Preprocesamiento")
from custom_transformers import OutlierHandler
import matplotlib.pyplot as plt
import plotly

In [182]:
SEMILLA = 42
FILENAME = "dataset_final.csv"
datapath = os.path.join("..","data","processed",FILENAME)
df = pd.read_csv(datapath)

In [183]:
y = df['target_Frio_kW_next']
df = df.drop(columns=['target_Frio_kW_next'])

In [184]:
with open ("../src/Preprocesamiento/preprocesamiento.pkl", "rb") as f:
    preprocessor = pickle.load(f)

In [185]:
df_preprocesado = preprocessor.transform(df)


### Separación de datos

In [186]:
X_train, X_test, y_train, y_test = train_test_split(df, y, train_size=0.7, shuffle=False, random_state=SEMILLA)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, train_size=0.5, shuffle=False, random_state=SEMILLA)


In [187]:
X_train = preprocessor.transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)

### Baselines: MAE

Día de hoy

In [188]:
print(type(preprocessor.named_steps['Escalado']))

<class 'sklearn.compose._column_transformer.ColumnTransformer'>


In [189]:
columns_no_ciclicas = X_train.columns[~X_train.columns.isin(["cos_month_of_year", "sin_month_of_year"])]
df_sin_escalar = (
    preprocessor.named_steps['Escalado']
    .named_transformers_['Escalado']
    .set_output(transform="pandas")
    .inverse_transform(X_val[columns_no_ciclicas])
)
df_sin_escalar = pd.DataFrame(df_sin_escalar, columns=columns_no_ciclicas)

In [190]:
y_val

805    16144.0
806    12401.0
807    15177.0
808    17596.0
809    17489.0
        ...   
972    20378.0
973    21941.0
974    22115.0
975    17634.0
976    22816.0
Name: target_Frio_kW_next, Length: 172, dtype: float64

In [191]:
y_frio_hoy = df_sin_escalar['Frio (Kw)'].values
MAE_frio_hoy = np.mean(np.abs(y_frio_hoy - y_val))
print(f"MAE entre Frio (Kw) y target_Frio_kW_next: {MAE_frio_hoy}")

MAE entre Frio (Kw) y target_Frio_kW_next: 2724.4767441860463


Promedio móvil

In [192]:
y_prom_movil = df_sin_escalar['Prom movil Frio'].values
MAE_prom_movil = np.mean(np.abs(y_prom_movil - y_val))
print(f"MAE entre Prom movil Frio y target_Frio_kW_next: {MAE_prom_movil:.4f}")

MAE entre Prom movil Frio y target_Frio_kW_next: 2393.5963


## Entrenamiento de modelos

### GXBoost

In [193]:
gxboost = GradientBoostingRegressor(loss="absolute_error" , random_state=SEMILLA)
gxboost.fit(X_train, y_train)
y_pred = gxboost.predict(X_val)
MAE_gxboost = np.mean(np.abs(y_pred - y_val))
print(f"MAE: {MAE_gxboost:.4f}")

MAE: 2176.7748


In [194]:
gxboost_verdad = XGBRegressor(random_state=SEMILLA)
gxboost_verdad.fit(X_train, y_train)
y_pred = gxboost_verdad.predict(X_val)
MAE_gxboost_verdad = np.mean(np.abs(y_pred - y_val))
print(f"MAE: {MAE_gxboost_verdad:.4f}")

MAE: 2697.8241


### Random Forest Regressor

In [195]:
randomforest = RandomForestRegressor(criterion="absolute_error",random_state=SEMILLA)
randomforest.fit(X_train, y_train)
MAE_randomforest = np.mean(np.abs(randomforest.predict(X_val) - y_val))
print(f"MAE: {MAE_randomforest:.4f}")

MAE: 2333.4922


### lightGBM

In [196]:
lightgbm = LGBMRegressor(objective="mae",random_state=SEMILLA,verbose=-1)
lightgbm.fit(X_train, y_train)
y_pred = lightgbm.predict(X_val)
MAE_lightgbm = np.mean(np.abs(y_pred - y_val))
print(f"MAE: {MAE_lightgbm:.4f}")

MAE: 2417.9880


### Lasso

In [197]:
lasso = Lasso(random_state=SEMILLA)
lasso.fit(X_train, y_train)
y_pred = lasso.predict(X_val)
MAE_lasso = np.mean(np.abs(y_pred - y_val))
print(f"MAE: {MAE_lasso:.4f}")

MAE: 2262.9929


In [198]:

# log_file = "results/experiments_log.csv"
# exp_id = "exp_01"
# if os.path.exists(log_file):
#     log_df = pd.read_csv(log_file,sep=',')
#     if not log_df.empty:
#         exp_id = log_df.iloc[-1]['experiment_id']
#         exp_id = int(exp_id.split('_')[-1]) + 1
#         exp_id = f"exp_{exp_id:02d}" 

# log_data = pd.DataFrame([{
#     "experiment_id": exp_id,
#     "datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
#     "model_version": "v1",
#     "hyperparameters": "RandomForest(n_estimators=100, random_state=42)",
#     "metrics": f"MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.2f}"
# }])

# if not os.path.exists(log_file):
#     log_data.to_csv(log_file, index=False,header=True)
# else:
#     log_data.to_csv(log_file, mode='a', header=False, index=False)

### Busqueda de hiperparámetros

### GXBoost

In [204]:
def objective_gxb(trial):
   n_estimators = trial.suggest_int("n_estimators", 30,250)
   max_depth = trial.suggest_int("max_depth", 1, 10)
   min_samples_split = trial.suggest_int("min_samples_split", 2,20)
   min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
   learning_rate = trial.suggest_float("learning_rate", 0.005, 0.2, log=True)
   alpha = trial.suggest_float("alpha", 0.01, 1)
   
   model = GradientBoostingRegressor(
       loss="absolute_error",
       n_estimators= n_estimators,
       max_depth= max_depth,
       min_samples_split= min_samples_split,
       min_samples_leaf= min_samples_leaf,
       learning_rate= learning_rate,
       alpha = alpha,
       random_state=SEMILLA
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   score = np.mean(np.abs(y_pred - y_val))
   return score
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction = "minimize", sampler=sampler)
study.optimize(objective_gxb, n_trials=100, n_jobs=1)

[I 2026-11-13 19:24:08,280] A new study created in memory with name: no-name-e3059d36-e9ed-415f-b9b3-db70cd0db9b7
[I 2026-11-13 19:24:09,243] Trial 0 finished with value: 3421.2195658135283 and parameters: {'n_estimators': 112, 'max_depth': 10, 'min_samples_split': 15, 'min_samples_leaf': 12, 'learning_rate': 0.00889039845957559, 'alpha': 0.16443457513284063}. Best is trial 0 with value: 3421.2195658135283.
[I 2026-11-13 19:24:09,524] Trial 1 finished with value: 6374.665320855333 and parameters: {'n_estimators': 42, 'max_depth': 9, 'min_samples_split': 13, 'min_samples_leaf': 15, 'learning_rate': 0.005394455304087533, 'alpha': 0.9702107536403743}. Best is trial 0 with value: 3421.2195658135283.
[I 2026-11-13 19:24:10,212] Trial 2 finished with value: 2406.6130996433744 and parameters: {'n_estimators': 213, 'max_depth': 3, 'min_samples_split': 5, 'min_samples_leaf': 4, 'learning_rate': 0.015359756451337138, 'alpha': 0.5295088673159155}. Best is trial 2 with value: 2406.6130996433744.
[

In [18]:
import optuna.visualization as vis
import plotly

In [205]:

fig_history = vis.plot_optimization_history(study)
fig_history.show()
fig_history = vis.plot_parallel_coordinate(study)
fig_history.show()
fig_contour = vis.plot_contour(study)
fig_contour.show()


fig_importance = vis.plot_param_importances(study)
fig_importance.show()

### RandomForest

In [175]:
def objective_rf(trial):
   n_estimators = trial.suggest_int("n_estimators", 50,300)
   max_depth = trial.suggest_int("max_depth", 1, 15)
   min_samples_split = trial.suggest_int("min_samples_split", 2,25)
   min_samples_leaf =  trial.suggest_int("min_samples_leaf", 1, 25)
   max_features = trial.suggest_float("max_features", 0.3, 1.0)

   
   
   model = RandomForestRegressor(
      criterion="absolute_error",
       n_estimators= n_estimators,
       max_depth= max_depth,
       min_samples_split= min_samples_split,
       min_samples_leaf= min_samples_leaf,
       max_features= max_features,
       random_state=SEMILLA
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   score = np.mean(np.abs(y_pred - y_val))
   return score

study_rf = optuna.create_study(direction = "minimize", sampler=sampler)
study_rf.optimize(objective_rf, n_trials=100)

[I 2026-11-13 18:56:07,286] A new study created in memory with name: no-name-a559e134-cac1-4428-8382-b84c7e5e97f4


[I 2026-11-13 18:56:07,913] Trial 0 finished with value: 2275.9216896060752 and parameters: {'n_estimators': 147, 'max_depth': 5, 'min_samples_split': 21, 'min_samples_leaf': 9, 'max_features': 0.49665415678116653}. Best is trial 0 with value: 2275.9216896060752.
[I 2026-11-13 18:56:08,911] Trial 1 finished with value: 2338.2103650912727 and parameters: {'n_estimators': 186, 'max_depth': 3, 'min_samples_split': 21, 'min_samples_leaf': 2, 'max_features': 0.9908208556203622}. Best is trial 0 with value: 2275.9216896060752.
[I 2026-11-13 18:56:10,171] Trial 2 finished with value: 2367.304299454493 and parameters: {'n_estimators': 243, 'max_depth': 3, 'min_samples_split': 2, 'min_samples_leaf': 21, 'max_features': 0.7948001406933319}. Best is trial 0 with value: 2275.9216896060752.
[I 2026-11-13 18:56:11,390] Trial 3 finished with value: 2255.8883570569365 and parameters: {'n_estimators': 232, 'max_depth': 12, 'min_samples_split': 3, 'min_samples_leaf': 9, 'max_features': 0.381108341667590

In [176]:

fig_history = vis.plot_optimization_history(study_rf)
fig_history.show()
fig_history = vis.plot_parallel_coordinate(study_rf)
fig_history.show()
fig_contour = vis.plot_contour(study_rf)
fig_contour.show()


fig_importance = vis.plot_param_importances(study_rf)
fig_importance.show()

### lightGBM

In [177]:
def objective_lgbm(trial):
   n_estimators = trial.suggest_int("n_estimators", 50,200)
   max_depth = trial.suggest_int("max_depth", 3, 20)
   learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
   reg_alpha = trial.suggest_float("reg_alpha", 0.0, 0.1)
   reg_lambda = trial.suggest_float("reg_lambda", 0.0, 0.1)
   
   model = LGBMRegressor(
       n_estimators= n_estimators,
       max_depth= max_depth,
       learning_rate= learning_rate,
       reg_alpha= reg_alpha,
       reg_lambda= reg_lambda,
       random_state=SEMILLA,
       verbose=-1
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   score = np.mean(np.abs(y_pred - y_val))
   return score

study_lgbm = optuna.create_study(direction = "minimize")
study_lgbm.optimize(objective_lgbm, n_trials=100)

[I 2026-11-13 18:58:20,163] A new study created in memory with name: no-name-949f61ee-2302-4c13-9ec6-5ea2cd64de44
[I 2026-11-13 18:58:20,257] Trial 0 finished with value: 2876.8787952124276 and parameters: {'n_estimators': 63, 'max_depth': 19, 'learning_rate': 0.023298638942387248, 'reg_alpha': 0.08659633713504519, 'reg_lambda': 0.09265746457823544}. Best is trial 0 with value: 2876.8787952124276.
[I 2026-11-13 18:58:20,302] Trial 1 finished with value: 2580.7909091406036 and parameters: {'n_estimators': 76, 'max_depth': 11, 'learning_rate': 0.08580390633782543, 'reg_alpha': 0.01046636484934358, 'reg_lambda': 0.04533664930123987}. Best is trial 1 with value: 2580.7909091406036.
[I 2026-11-13 18:58:20,334] Trial 2 finished with value: 2573.0033860066765 and parameters: {'n_estimators': 141, 'max_depth': 4, 'learning_rate': 0.07655955803428724, 'reg_alpha': 0.06292180403796593, 'reg_lambda': 0.011302121723788706}. Best is trial 2 with value: 2573.0033860066765.
[I 2026-11-13 18:58:20,410

In [178]:

fig_history = vis.plot_optimization_history(study_lgbm)
fig_history.show()
fig_history = vis.plot_parallel_coordinate(study_lgbm)
fig_history.show()
fig_contour = vis.plot_contour(study_lgbm)
fig_contour.show()


fig_importance = vis.plot_param_importances(study_lgbm)
fig_importance.show()

### Lasso

In [179]:
def objective_lasso(trial):
   alpha = trial.suggest_float("alpha", 0.1, 5000, log=True)
   
   model = Lasso(
       alpha= alpha,
       random_state=SEMILLA,
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   score = np.mean(np.abs(y_pred - y_val))
   return score

study_lasso = optuna.create_study(direction = "minimize")
study_lasso.optimize(objective_lasso, n_trials=100)

[I 2026-11-13 18:58:51,752] A new study created in memory with name: no-name-c7481952-6538-4367-8624-88357578d50f
[I 2026-11-13 18:58:51,760] Trial 0 finished with value: 2282.179555721728 and parameters: {'alpha': 0.10666173469477275}. Best is trial 0 with value: 2282.179555721728.
[I 2026-11-13 18:58:51,765] Trial 1 finished with value: 2207.944279481648 and parameters: {'alpha': 179.4628727053064}. Best is trial 1 with value: 2207.944279481648.
[I 2026-11-13 18:58:51,770] Trial 2 finished with value: 2254.588586411591 and parameters: {'alpha': 14.328262312234651}. Best is trial 1 with value: 2207.944279481648.
[I 2026-11-13 18:58:51,777] Trial 3 finished with value: 2986.9000784045706 and parameters: {'alpha': 1456.7156591816895}. Best is trial 1 with value: 2207.944279481648.
[I 2026-11-13 18:58:51,782] Trial 4 finished with value: 2235.201358960093 and parameters: {'alpha': 27.066668572005238}. Best is trial 1 with value: 2207.944279481648.
[I 2026-11-13 18:58:51,786] Trial 5 fini

In [180]:

fig_history = vis.plot_optimization_history(study_lasso)
fig_history.show()
fig_history = vis.plot_parallel_coordinate(study_lasso)
fig_history.show()
fig_contour = vis.plot_contour(study)
fig_contour.show()


fig_importance = vis.plot_param_importances(study)
fig_importance.show()

In [73]:
def objective_gxboost(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 800)
    max_depth = trial.suggest_int("max_depth", 3, 9)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.2, log=True)
    
    # Regularización y muestreo para evitar overfitting
    min_child_weight = trial.suggest_int("min_child_weight", 1, 10)
    subsample = trial.suggest_float("subsample", 0.6, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0)
    
    reg_alpha = trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True)

    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        min_child_weight=min_child_weight,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=SEMILLA,
        eval_metric="mae"   # Métrica interna coherente con tu evaluación
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    
    score = np.mean(np.abs(y_pred - y_val))
    return score

study_gxboost = optuna.create_study(direction="minimize")
study_gxboost.optimize(objective_gxboost, n_trials=100)

[I 2026-11-13 17:59:22,074] A new study created in memory with name: no-name-c925b234-12a2-40e8-9e26-096bed3b50b2
[I 2026-11-13 17:59:23,476] Trial 0 finished with value: 2504.489734738372 and parameters: {'n_estimators': 282, 'max_depth': 8, 'learning_rate': 0.01322667337697526, 'min_child_weight': 4, 'subsample': 0.7131870265144629, 'colsample_bytree': 0.8534032994774376, 'reg_alpha': 0.0012955716071708118, 'reg_lambda': 0.0026535755691098365}. Best is trial 0 with value: 2504.489734738372.
[I 2026-11-13 17:59:24,006] Trial 1 finished with value: 2790.228819381359 and parameters: {'n_estimators': 359, 'max_depth': 9, 'learning_rate': 0.1925709019546521, 'min_child_weight': 9, 'subsample': 0.9670831526019583, 'colsample_bytree': 0.6264828270239026, 'reg_alpha': 6.675707456089141, 'reg_lambda': 2.8972327543729772}. Best is trial 0 with value: 2504.489734738372.
[I 2026-11-13 17:59:24,284] Trial 2 finished with value: 2876.423708893532 and parameters: {'n_estimators': 578, 'max_depth': 